# Bölüm 7: Sohbet Uygulamaları Oluşturma
## OpenAI API Hızlı Başlangıç

Bu not defteri, [Azure OpenAI Örnekleri Deposu](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) üzerinden uyarlanmıştır ve içinde [Azure OpenAI](notebook-azure-openai.ipynb) hizmetlerine erişen not defterleri bulunmaktadır.

Python OpenAI API, birkaç değişiklikle Azure OpenAI Modelleriyle de çalışır. Farklar hakkında daha fazla bilgi edinin: [Python ile OpenAI ve Azure OpenAI uç noktaları arasında geçiş yapma](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Genel Bakış  
"Büyük dil modelleri, metni metne eşleyen fonksiyonlardır. Verilen bir giriş metni dizisi için, büyük dil modeli sonraki gelecek metni tahmin etmeye çalışır"(1). Bu "hızlı başlangıç" defteri, kullanıcılara üst düzey LLM kavramlarını, AML ile başlamak için gerekli temel paketleri, prompt tasarımına yumuşak bir giriş ve farklı kullanım durumlarının birkaç kısa örneğini tanıtacaktır. 


## İçindekiler  

[Genel Bakış](#overview)  
[OpenAI Servisi Nasıl Kullanılır](#how-to-use-openai-service)  
[1. OpenAI Servisinizi Oluşturma](#1.-creating-your-openai-service)  
[2. Kurulum](#2.-installation)    
[3. Kimlik Bilgileri](#3.-credentials)  

[Kullanım Durumları](#use-cases)    
[1. Metni Özetle](#1.-summarize-text)  
[2. Metni Sınıflandır](#2.-classify-text)  
[3. Yeni Ürün İsimleri Oluştur](#3.-generate-new-product-names)  
[4. Bir Sınıflandırıcıyı İnce Ayar Yap](#4.fine-tune-a-classifier)  

[Referanslar](#references)


### İlk promptunuzu oluşturun  
Bu kısa egzersiz, basit bir görev olan "özetleme" için bir OpenAI modeline prompt göndermeye temel bir giriş sağlayacaktır.


**Adımlar**:  
1. Python ortamınıza OpenAI kütüphanesini kurun  
2. Standart yardımcı kütüphaneleri yükleyin ve oluşturduğunuz OpenAI Hizmeti için tipik OpenAI güvenlik kimlik bilgilerinizi ayarlayın  
3. Göreviniz için bir model seçin  
4. Model için basit bir prompt oluşturun  
5. İsteğinizi model API'sine gönderin!


### 1. OpenAI Kurulumu


In [ ]:
%pip install openai python-dotenv

### 2. Yardımcı kütüphaneleri içe aktar ve kimlik bilgilerini oluştur


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Doğru modeli bulmak  
GPT-4o ve GPT-4o mini gibi modeller doğal dili anlayabilir ve üretebilir ve farklı görevler için uygun farklı güç ve hız seviyeleriyle OpenAI platformunda mevcuttur.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. İstek (Prompt) Tasarımı  

"Büyük dil modellerinin sihri, bu tahmin hatasını çok büyük miktarda metin üzerinde minimize edecek şekilde eğitilerek, modellerin bu tahminler için faydalı kavramları öğrenmesiyle ortaya çıkar. Örneğin, aşağıdaki kavramları öğrenirler"(1):

* nasıl yazılır
* dilbilgisinin nasıl işlediği
* nasıl yeniden ifade edilir
* nasıl sorulara cevap verilir
* nasıl konuşma sürdürülür
* birçok dilde nasıl yazılır
* nasıl kod yazılır
* vb.

#### Büyük bir dil modeli nasıl kontrol edilir  
"Bir büyük dil modeline verilen tüm girdiler arasında, en etkili olanı açık ara metin istemidir (prompt)(1).

Büyük dil modellerinden çıktı üretilmesi birkaç şekilde yapılabilir:

Talimat: Modele ne istediğinizi söyleyin
Tamamlama: Modeli istediğiniz şeyin başlangıcını tamamlamaya yönlendirin
Gösterim: Modele ne istediğinizi gösterin, şu şekilde:
İstemde birkaç örnek
İnce ayar eğitim veri setinde yüzlerce hatta binlerce örnek"



#### İstek oluşturmak için üç temel kural vardır:

**Göster ve anlat**. Ne istediğinizi talimatlar, örnekler veya ikisinin birleşimi ile net bir şekilde belirtin. Modelin bir listeyi alfabetik sıraya koymasını ya da bir paragrafı duygu durumuna göre sınıflandırmasını isterseniz, ona bunu göstermek gerek.

**Kaliteli veri sağlayın**. Bir sınıflandırıcı oluşturuyorsanız veya modelin bir kalıbı takip etmesini istiyorsanız, yeterli örnek olduğundan emin olun. Örneklerinizi mutlaka gözden geçirin — model genellikle temel yazım hatalarını fark edecek kadar zekidir ve size yanıt verebilir, ancak bunun bilinçli olduğunu varsayabilir ve bu yanıtı etkileyebilir.

**Ayarlarınızı kontrol edin.** Temperature ve top_p ayarları, modelin yanıt oluştururken ne kadar deterministik olduğunu kontrol eder. Eğer sadece bir doğru cevabın olduğu bir yanıt istiyorsanız, bu ayarları daha düşük tutmak istersiniz. Daha çeşitli yanıtlar arıyorsanız, bu ayarları daha yüksek yapabilirsiniz. İnsanların bu ayarlarla yaptığı bir numaralı hata, bunların "zekâ" veya "yaratıcılık" ayarları olduğunu sanmaktır.


Kaynak: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Gönder!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Aynı çağrıyı tekrarlayın, sonuçlar nasıl karşılaştırılır? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Metni Özetle  
#### Zorluk  
Bir metin pasajının sonuna 'tl;dr:' ekleyerek metni özetleyin. Modelin ek talimatlar olmadan nasıl çeşitli görevleri yerine getirebildiğini fark edin. Modelin davranışını değiştirmek ve aldığınız özetlemeyi kişiselleştirmek için tl;dr'den daha açıklayıcı istemlerle deneyler yapabilirsiniz(3).  

Son zamanlardaki çalışmalar, belirli bir görevde ince ayar yapılmadan önce büyük bir metin korpusunda ön eğitim yaparak birçok NLP görevi ve kıyas testinde önemli gelişmeler göstermiştir. Genellikle mimarisi açısından görevden bağımsız olsa da, bu yöntem hâlâ binlerce veya on binlerce örnekten oluşan görev-özel ince ayar veri setlerine ihtiyaç duymaktadır. Buna karşılık, insanlar genellikle sadece birkaç örnekten veya basit talimatlardan yeni bir dil görevini yerine getirebilirler - ki mevcut NLP sistemlerinin büyük ölçüde başarmakta zorlandığı bir şey. Burada, dil modellerini ölçeklendirmenin görevden bağımsız, az örnekli performansı büyük ölçüde iyileştirdiğini, bazen önceki en iyi ince ayar yaklaşımlarıyla rekabet edilebilir seviyelere ulaştığını gösteriyoruz. 



Özet  


# Birkaç kullanım durumu için alıştırmalar  
1. Metni Özetle  
2. Metni Sınıflandır  
3. Yeni Ürün İsimleri Üret


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Metni Sınıflandır  
#### Meydan Okuma  
Öğeleri çıkarım zamanında sağlanan kategorilere sınıflandırın. Aşağıdaki örnekte, hem kategorileri hem de sınıflandırılacak metni istemde (*playground_reference) sağlıyoruz. 

Müşteri Sorgusu: Merhaba, dizüstü bilgisayarımın klavyesindeki bir tuş yakın zamanda kırıldı ve bir yedeğe ihtiyacım olacak:

Sınıflandırılmış kategori:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Yeni Ürün İsimleri Oluşturma
#### Zorluk
Örnek kelimelerden ürün isimleri oluşturun. Burada, isim oluşturacağımız ürün hakkında bilgi sağlıyoruz. Aynı zamanda almak istediğimiz örüntüyü göstermek için benzer bir örnek veriyoruz. Rastgeleliği ve daha yenilikçi yanıtları artırmak için sıcaklık değerini yüksek olarak ayarladık.

Ürün açıklaması: Ev tipi milkshake yapıcı
Tohum kelimeler: hızlı, sağlıklı, kompakt.
Ürün isimleri: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Ürün açıklaması: Her ayak numarasına uygun bir çift ayakkabı.
Tohum kelimeler: uyarlanabilir, uygun, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referanslar  
- [Openai Yemek Kitabı](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portalı](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Metni sınıflandırmak için GPT modellerini ince ayar yapmanın en iyi uygulamaları](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Daha Fazla Yardım İçin  
[OpenAI Ticari Çalışma Ekibi](AzureOpenAITeam@microsoft.com) 


# Katkıda Bulunanlar
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
